# Mission 2: 신고자 / 119대원 발화 분류 (발화 조각 음성 → speaker)

Colab(GPU) 에서 위에서부터 순서대로 실행하세요. 실행 로그가 그대로 학습 로그로 제출됩니다.

In [ ]:
# ① GPU 확인 + Google Drive 연결 + 코드 받기
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/DCC'):
    !git clone -q -b ccr-1cdc76be-ibbw6s https://github.com/BioCode67/DCC.git /content/DCC
else:
    !git -C /content/DCC pull -q


In [ ]:
!pip -q install -r /content/DCC/mission2/requirements.txt

In [ ]:
# ② 데이터 폴더 찾기
#   공유 폴더 '2026 DCC 대학부 데이터' 를 내 드라이브에 '바로가기 추가' 해 두면 아래 경로로 보임.
#   경로가 다르면 DATA_ROOT 만 고쳐주세요.
import glob, os, zipfile
DATA_ROOT = '/content/drive/MyDrive/2026 학습/2026 DCC 대학부 데이터'
if not os.path.exists(DATA_ROOT):
    cands = glob.glob('/content/drive/MyDrive/**/2026 DCC 대학부 데이터', recursive=True)
    DATA_ROOT = cands[0] if cands else DATA_ROOT
print('DATA_ROOT =', DATA_ROOT, os.path.exists(DATA_ROOT))

def find_dir(prefix):
    """TS_/TL_/VS_/VL_ 로 시작하는 폴더(또는 zip)를 찾아 경로 반환. zip 이면 로컬 디스크에 압축 해제."""
    hits = sorted(glob.glob(f'{DATA_ROOT}/**/{prefix}*', recursive=True))
    for h in hits:
        if os.path.isdir(h):
            return h
    for h in hits:
        if h.endswith('.zip'):
            dst = f'/content/data/{os.path.basename(h)[:-4]}'
            if not os.path.exists(dst):
                print('unzip', h); zipfile.ZipFile(h).extractall(dst)
            return dst
    return None

TRAIN_WAV, TRAIN_JSON = find_dir('TS_'), find_dir('TL_')
VAL_WAV, VAL_JSON = find_dir('VS_'), find_dir('VL_')
for k, v in dict(TRAIN_WAV=TRAIN_WAV, TRAIN_JSON=TRAIN_JSON, VAL_WAV=VAL_WAV, VAL_JSON=VAL_JSON).items():
    n = len(glob.glob(f'{v}/**/*.*', recursive=True)) if v else 0
    print(f'{k:10s} {v}  ({n} files)')


In [ ]:
# ③ 결과 저장 위치 (런타임이 끊겨도 남도록 Drive 에 저장. 용량이 부족하면 /content/ckpt 로 바꾸고 직접 다운로드)
SAVE_DIR = '/content/drive/MyDrive/DCC_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)
QUICK_TEST = 0   # 처음엔 200 정도로 전체 흐름만 확인 → 문제 없으면 0(전체 데이터)으로 다시 실행


In [ ]:
# ④ 학습 (WavLM 특징 추출 → MLP 헤드 학습, Training 폴더만 사용)
import sys, time; sys.path.insert(0, '/content/DCC/mission2'); os.chdir('/content/DCC/mission2')
import train
args = train.get_parser().parse_args([
    '--train_audio_dir', TRAIN_WAV, '--train_label_dir', TRAIN_JSON,
    '--ckpt_path', f'{SAVE_DIR}/mission2.pt', '--cache_dir', '/content/cache',
    '--num_workers', '8', '--limit', str(QUICK_TEST)])
t0 = time.time(); train.main(args); print(f'총 학습 시간 {time.time()-t0:.0f}s')


In [ ]:
# ⑤ Validation 추론 (제출 명령어와 동일) + 채점 + 추론 시간 측정
!cd /content/DCC/mission2 && time python inference.py --audio_dir "$VAL_WAV" --label_dir "$VAL_JSON" \
    --ckpt_path "$SAVE_DIR/mission2.pt" --output ./outputs/mission2.csv --num_workers 8
!python /content/DCC/tools/evaluate.py --mission 2 --pred /content/DCC/mission2/outputs/mission2.csv --label_dir "$VAL_JSON"
!cp /content/DCC/mission2/outputs/mission2.csv "$SAVE_DIR/"
